# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa – Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library, which provides a standard interface for datasets described with the [Croissant](https://mlcommons.org/croissant) metadata schema.

### Dataset Source
The dataset is loaded from its Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset's Croissant metadata using `mlcroissant` to programmatically explore and interact with its record sets and fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"
dataset = mlc.Dataset(croissant_url)

# View summary metadata (access as a dataclass, use dot notation)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}\n")
print(f"Version: {metadata.version}\n")
print(f"Keywords: {', '.join(metadata.keywords) if hasattr(metadata, 'keywords') else 'N/A'}\n")

## 2. Data Overview

Let's enumerate the available record sets, their fields, and IDs using the Croissant metadata structure and the mlcroissant Python interface. Each record set, field, and column is identified by its `@id` field.

In [ ]:
# List all record sets and their associated fields and columns
print("Available record sets (@id, name, description):\n")
record_set_ids = []

# 'record_sets' is mlcroissant's list of RecordSet objects from the dataset
for rs in dataset.record_sets():
    print(f"@id: {rs.id}\n  name: {rs.name}\n  description: {rs.description if hasattr(rs, 'description') else 'No description'}\n")
    record_set_ids.append(rs.id)
    # List top-level fields for each record set
    print(f" Fields and Columns in {rs.id}:")
    for field in rs.fields:
        print(f"   [field @id: {field.id}] {field.name} - type: {getattr(field, 'data_type', None)}")
        if hasattr(field, 'source_column') and field.source_column is not None:
            col = field.source_column
            # Source column is a Column object
            print(f"     column @id: {col.id}, name: {col.name}, type: {col.data_type if hasattr(col, 'data_type') else None}")
    print()

# Use the first record set by default for further examples
if record_set_ids:
    default_record_set_id = record_set_ids[0]
else:
    default_record_set_id = None

## 3. Data Extraction

Let's extract data from each record set using their `@id` with the `dataset.records(record_set=...)` API. The resulting records are loaded as Pandas DataFrames for analysis.

In [ ]:
# Build a dict of all DataFrames (one per record set)
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record_set: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}\n")

# Preview data from the first record set
if default_record_set_id is not None:
    print(f"Data preview from record set {default_record_set_id}:")
    display(dataframes[default_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's perform several common data processing and analysis steps:
- Filter records based on a numeric field using its `@id`
- Normalize the chosen field
- Group data by a categorical field (e.g., `gender` or similar)

First, let's list the numeric fields in the default record set for user selection.

In [ ]:
from IPython.display import display

# Find a numeric field for EDA (search for fields with Integer/Float data type)
rs = next((rs for rs in dataset.record_sets() if rs.id == default_record_set_id), None)
numeric_fields = []
groupable_fields = []

if rs is not None:
    for field in rs.fields:
        dt = getattr(field, 'data_type', None)
        if dt in ['Integer', 'Float', 'Number']:
            numeric_fields.append((field.id, field.name, dt))
        # Consider string fields for grouping, like 'gender'
        if dt == 'Text' and ('gender' in field.name.lower() or 'village' in field.name.lower() or 'education' in field.name.lower()):
            groupable_fields.append((field.id, field.name, dt))

    print("Numeric fields:")
    for fid, fname, dt in numeric_fields:
        print(f"  @id: {fid} | name: {fname} | type: {dt}")

    print("Groupable string fields (for grouping/categorization):")
    for fid, fname, dt in groupable_fields:
        print(f"  @id: {fid} | name: {fname} | type: {dt}")

# Select the first available numeric field and groupable field for demonstration
if numeric_fields:
    numeric_field_id, numeric_field_name, _ = numeric_fields[0]
else:
    numeric_field_id = None
    numeric_field_name = None

if groupable_fields:
    group_field_id, group_field_name, _ = groupable_fields[0]
else:
    group_field_id = None
    group_field_name = None

print(f"\nUsing numeric field for filtering: {numeric_field_id} ({numeric_field_name})")
print(f"Grouping by field: {group_field_id} ({group_field_name})")

In [ ]:
# --- EDA Filtering, Normalizing, Grouping ---

df = dataframes[default_record_set_id].copy() if default_record_set_id is not None else pd.DataFrame()

if numeric_field_id is not None and numeric_field_id in df.columns:
    # Drop missing or invalid entries
    df = df.dropna(subset=[numeric_field_id])
    # Robust filtering threshold
    try:
        threshold = df[numeric_field_id].quantile(0.75)
    except Exception:
        threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (field name: {numeric_field_name}):\n")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field if available
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA in default record set.")

## 5. Visualization

Visualize the distribution of the selected numeric field using matplotlib and seaborn. This helps reveal patterns or outliers in the data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id} ({numeric_field_name})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion

Using the Croissant schema and the `mlcroissant` Python library, we have programmatically loaded the Kilifi County Mental Health Survey dataset, examined its structure, and performed basic filtering, normalization, grouping, and visualization. This foundation supports more advanced analytics and AI/ML workflows, built on open standards for dataset interoperability.

For more information about the schema, please refer to the [FAIR² dataset page](https://sen.science/doi/10.71728/senscience.vcs2-05nj).

_Note: All record sets, fields, and columns are referenced by their Croissant `@id` for clarity and interoperability._